# Compaction 

### source : COMPACTION_6/compaction_2.ipynb


### Quick reminder of what compaction is. 
When a conversation gets long, you write a summary of it, and restart the assistant from that summary. 
The skill is choosing what to keep and what to throw away.

## Setup
### Load credentials and create the summarising model

We read `DATABRICKS_HOST` and `DATABRICKS_TOKEN` from a `.env` file so nothing is hard-coded. 


In [ ]:
import os
from dotenv import load_dotenv

# Load credentials from .env at project root
load_dotenv()

# Verify the credentials are present
assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set — check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set — check your .env file"

In [ ]:
from databricks_langchain import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # confirmed Free-Edition model

def compactor(max_summary_tokens: int) -> ChatDatabricks:
    """A summarising model with a fixed summary-length budget (see §6.6).
    temperature=0 keeps the output stable so the evaluation below is repeatable."""
    return ChatDatabricks(endpoint=ENDPOINT, temperature=0.0, max_tokens=max_summary_tokens)


### A sample conversation: planning a trip to Lisbon

A good test transcript mixes **things we must keep** with **things we can safely throw
away**. 



In [ ]:
conversation: list[BaseMessage] = [
    HumanMessage("Plan a 5-day trip to Lisbon in May. Budget: 1200 euros total. "
                 "Please only book hotels that offer free cancellation."),
    AIMessage("I'd absolutely love to help you plan this!"),                       # Polite ack -> DISCARD
    AIMessage("Decision: I'll base you in the Alfama neighbourhood, so most sights "
              "are within walking distance."),                                     # Decision  -> KEEP
    HumanMessage("What's the weather like then?"),
    AIMessage("Checked a forecast: Lisbon in May averages about 21 C with little rain."),  
    AIMessage("Booked Hotel do Convento - free cancellation, 95 euros/night. "
              "Confirmation code LX48213."),                                       # decision + reference -> KEEP
    AIMessage("Just re-checked the room is still available - yes, all good."),      # redundant retry -> DISCARD
    HumanMessage("Can we add a day trip to Sintra?"),
    AIMessage("Open issue: Pena Palace is closed on Mondays, and your only free day "
              "is a Monday. This still needs resolving."),                         # open issue -> KEEP
    AIMessage("Okay, understood - I'll look into alternatives."),                  # filler ack -> DISCARD
]

print(f"{len(conversation)} messages in the raw transcript.")

## 1- Compaction

## 1-1 - Naive trimming 

The simplest "compaction" just deletes the oldest messages (first-in, first-out).
It's cheap and predictable, but watch what happens: the goal and the rules were stated
*first*, so they're the *first* things thrown away.

In [ ]:
def fifo_trim(messages: list[BaseMessage], keep_last: int) -> list[BaseMessage]:
    """Keep only the most recent `keep_last` messages."""
    return messages[-keep_last:]


In [ ]:
trimmed = fifo_trim(conversation, keep_last=6)
trim_text = "\n".join(f"[{m.type}] {m.content}" for m in trimmed)
print("After FIFO trimming, only these remain:")
for m in trimmed:
    print(f"  [{m.type}] {m.content[:70]}")
# The budget, the free-cancellation rule and the booking code are all gone.

### 1-2 - Aggressive and conservative compaction

Real compaction uses the model to summarise instead. The instructions below are
the **keep / summarise / discard** rules written out for the
model.

In [ ]:
# What we keep and discard
COMPACTION_SYSTEM_PROMPT = """\
Summarise this conversation so the assistant can carry on smoothly.

MUST KEEP (accurately):
- The original goal and any rules the user set
- Decisions already made
- Issue still unresolved or open
- Specific references: names, codes, numbers, dates

CAN SHORTEN (high-level): results already acted on, successful steps and their outcome.

DISCARD: pleasantries, repeated re-checks, filler acknowledgements.

Write the summary inside <summary></summary> tags. Text only; do not use tools."""

In [ ]:
def compact(messages: list[BaseMessage], model, system_message: str) -> str:
    """Distil a transcript into a short continuity summary."""
    transcript = "\n".join(f"[{m.type}] {m.content}" for m in messages)
    return model.invoke([
        SystemMessage(system_message),
        HumanMessage(f"Conversation:\n\n{transcript}"),
    ]).content

Aggressive vs conservative, side by side

Exactly the same prompt and transcript. The **only** thing we change is the length
budget (`max_summary_tokens` = max output tokens):

> Note: in production the study guide quotes ~500–1,000 tokens (aggressive) and
> ~2,000–4,000 (conservative). We use smaller values here because the example
> conversation is short.

Run the cell and read both — the difference is meant to be visible to the eye before we
even measure it.


In [ ]:
aggressive_summary   = compact(conversation, compactor(max_summary_tokens=30), system_message=COMPACTION_SYSTEM_PROMPT)
conservative_summary = compact(conversation, compactor(max_summary_tokens=80), system_message=COMPACTION_SYSTEM_PROMPT)

print("=== AGGRESSIVE (short budget) ===\n")
print(aggressive_summary)
print("\n=== CONSERVATIVE (long budget) ===\n")
print(conservative_summary)

## 2- Evaluation
## 2-1 - Measuring quality: recall first, then precision

The guide tunes compaction in two phases, and we can measure each one:

* **Phase 1 — Recall.** Did we keep *everything critical*? We check for "needles":
  facts that must survive. Missing needle = lost information = bad.
* **Phase 2 — Precision.** Did we *avoid keeping the junk*? We check that the
  discardable noise did **not** survive, and how short the summary is. Junk that
  survives = wasted tokens = low precision.

A good compaction scores **high on both**. Aggressive usually wins on precision but can
lose recall; conservative usually wins on recall but scores lower on precision.


In [ ]:
# Critical facts that MUST survive (recall).
NEEDLES = {
    "goal: Lisbon trip":          "lisbon",
    "rule: free cancellation":    "free cancellation",
    "rule: 1200 budget":          "1200",
    "decision: Alfama":           "alfama",
    "reference: booking code":    "lx48213",
    "open issue: Monday closure": "monday",
}

# Noise that should NOT survive (precision).
NOISE = {
    "pleasantry":      "love to help",
    "redundant recheck": "re-checked",
    "filler ack":      "understood",
}


In [ ]:
# Compactions
d_context_window_compaction = {
    "trimmed":trim_text,
    "aggressive_summary":aggressive_summary, 
    "conservative_summary":conservative_summary
}

def evaluate(summary: str) -> dict:
    #Evaluate recall and precision
    low = summary.lower()
    recall_hits    = sum(tok in low for tok in NEEDLES.values())
    precision_hits = sum(tok not in low for tok in NOISE.values())  # absent = good
    return {
        "words":     len(summary.split()),
        "recall":    recall_hits / len(NEEDLES),      # 1.0 = kept everything critical
        "precision": precision_hits / len(NOISE),     # 1.0 = dropped all the junk
    }

def show(summary: str, label: str) -> dict:
    #Present recall and precision result
    s = evaluate(summary)
    print(f"{label:<14} words={s['words']:>4}   "
          f"recall={s['recall']:.0%}   precision={s['precision']:.0%}")
    return s

In [ ]:
print("Scorecard (higher recall AND precision = better):\n")

for label, summary in d_context_window_compaction.items():
    #print(summary)
    show(summary, label.upper())

#### Where the points were lost

The scorecard gives the headline numbers; this cell shows *which* specific needle was
dropped or *which* piece of noise slipped through — the per-item view you use in
Phase 2 to tighten the prompt.

In [ ]:
def diagnose(summary: str, label: str) -> None:
    low = summary.lower()
    print(f"--- {label} ---")
    for name, tok in NEEDLES.items():
        print(f"  recall   {'KEPT   ' if tok in low else 'LOST !!'}  {name}")
    for name, tok in NOISE.items():
        print(f"  precision {'clean  ' if tok not in low else 'LEAKED!'}  {name}")
    print()

diagnose(aggressive_summary,   "Aggressive")
diagnose(conservative_summary, "Conservative")

### 2-2 - Restart the assistant from the summary

Compaction finishes by *replacing* the long history with the summary and carrying on.
We use the conservative summary (higher recall) so the assistant still knows the rules.

In [ ]:
def reinitialise(summary: str, next_message: str) -> list[BaseMessage]:
    """Swap the full history for the summary, then add the next turn (6.1)."""
    return [
        SystemMessage("You are continuing a trip-planning task. Here is the compacted "
                      f"context so far:\n\n{summary}"),
        HumanMessage(next_message),
    ]

assistant = ChatDatabricks(endpoint=ENDPOINT, temperature=0.2, max_tokens=400)


In [ ]:
turn = reinitialise(conservative_summary,
                    "Remind me my budget and what's still unresolved about Sintra.")
print("conservative summary :", assistant.invoke(turn).content)

In [ ]:
turn = reinitialise(aggressive_summary,
                    "Remind me my budget and what's still unresolved about Sintra.")
print("Aggressive summary :", assistant.invoke(turn).content)

In [ ]:
turn = reinitialise(trim_text,
                    "Remind me my budget and what's still unresolved about Sintra.")
print("Naive trimming : ", assistant.invoke(turn).content)